# LVIS Persona Study — Political Signal from Consumer Interests

180 personas, 9 groups (N=20), 6 ranking domains (3 Dem + 3 Rep items each), gpt-4o-mini.

In [ ]:
import json
import numpy as np
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['DejaVu Serif'],
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 7,
    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False,
})

ROOT = Path('/project/jevans/tzhang3/dotty-project/linear-political-llm')
FIGS = ROOT / 'figs' / 'lvis_persona'
FIGS.mkdir(parents=True, exist_ok=True)

GROUP_ORDER = ['baseline',
               'dem_impl_no_img', 'dem_impl_img',
               'rep_impl_no_img', 'rep_impl_img',
               'dem_explicit', 'rep_explicit',
               'dem_img_only', 'rep_img_only']

KEY_GROUPS = ['baseline',
              'dem_impl_no_img', 'rep_impl_no_img',
              'dem_explicit', 'rep_explicit',
              'dem_img_only', 'rep_img_only']

DEM_DARK  = '#1565A8'
DEM_LIGHT = '#5B9BD5'
REP_DARK  = '#B71C1C'
REP_LIGHT = '#E57373'
GRAY      = '#7F7F7F'
VIS_LIGHT = '#AB47BC'

def save_both(name):
    for ext in ('pdf', 'png'):
        plt.savefig(FIGS / f'{name}.{ext}', bbox_inches='tight', pad_inches=0.1)
    print(f'  {name}.pdf + .png')

def load_domain(domain):
    path = ROOT / 'data' / 'lvis_persona' / f'lvis_{domain}_classified.jsonl'
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


In [ ]:
DOMAIN_ITEMS = {
    'voting':   ['Chen (D)', 'Thompson (D)', 'Kowalski (D)',
                 'Mitchell (R)', 'Caldwell (R)', 'Wheeler (R)'],
    'politics': ['Climate', 'Healthcare', 'Education',
                 'Natl Security', 'Border', 'Gun Rights'],
    'books':    ['New Jim Crow', 'This Changes', 'Sixth Extinction',
                 '12 Rules', 'Hillbilly Elegy', 'American Sniper'],
    'weekend':  ['Farmers+Yoga', 'Art Gallery', 'Community Garden',
                 'Church Potluck', 'Shooting Range', 'BBQ+Cornhole'],
    'beer':     ['Hazy IPA', 'Kombucha', 'Non-alc IPA',
                 'Coors', 'Bud Light', 'Miller High Life'],
    'cars':     ['Prius', 'Model 3', 'Outback',
                 'F-150', 'Wrangler', 'Silverado'],
}

DEM_ITEMS = {
    'voting': ['Chen (D)', 'Thompson (D)', 'Kowalski (D)'],
    'politics': ['Climate', 'Healthcare', 'Education'],
    'books': ['New Jim Crow', 'This Changes', 'Sixth Extinction'],
    'weekend': ['Farmers+Yoga', 'Art Gallery', 'Community Garden'],
    'beer': ['Hazy IPA', 'Kombucha', 'Non-alc IPA'],
    'cars': ['Prius', 'Model 3', 'Outback'],
}
REP_ITEMS = {
    'voting': ['Mitchell (R)', 'Caldwell (R)', 'Wheeler (R)'],
    'politics': ['Natl Security', 'Border', 'Gun Rights'],
    'books': ['12 Rules', 'Hillbilly Elegy', 'American Sniper'],
    'weekend': ['Church Potluck', 'Shooting Range', 'BBQ+Cornhole'],
    'beer': ['Coors', 'Bud Light', 'Miller High Life'],
    'cars': ['F-150', 'Wrangler', 'Silverado'],
}

DOMAINS = ['voting', 'politics', 'books', 'weekend', 'beer', 'cars']

ranking_data = {}
for domain in DOMAINS:
    records = load_domain(domain)
    if not records:
        print(f'WARNING: no data for {domain}')
        continue
    items = list(records[0].get('_ranking', {}).keys())
    gv = defaultdict(lambda: defaultdict(list))
    for r in records:
        for item, rank in r.get('_ranking', {}).items():
            if rank is not None:
                gv[r['group']][item].append(rank)
    matrix = np.full((len(GROUP_ORDER), len(items)), np.nan)
    for gi, g in enumerate(GROUP_ORDER):
        for ii, item in enumerate(items):
            vals = gv[g].get(item, [])
            if vals:
                matrix[gi, ii] = np.mean(vals)
    ranking_data[domain] = {'records': records, 'items': items, 'matrix': matrix, 'gv': gv}
    print(f'{domain}: {len(records)} records')

print('\nLoaded.')

---
## Figure 1 — Pro-Dem Alignment by Group and Domain

`mean(Rep-item rank) − mean(Dem-item rank)`. Higher = more pro-Dem. When the model ranks Dem items better (lower rank) and Rep items worse (higher rank), the gap grows positive.

In [ ]:
alignment = {}
for domain in ranking_data:
    gv = ranking_data[domain]['gv']
    actual_items = ranking_data[domain]['items']
    short_items = DOMAIN_ITEMS[domain]
    lookup = dict(zip(short_items, actual_items))
    dem_actual = [lookup[d] for d in DEM_ITEMS[domain]]
    rep_actual = [lookup[r] for r in REP_ITEMS[domain]]

    group_align = {}
    for g in KEY_GROUPS:
        dem_ranks = sum((gv[g].get(i, []) for i in dem_actual), [])
        rep_ranks = sum((gv[g].get(i, []) for i in rep_actual), [])
        if dem_ranks and rep_ranks:
            group_align[g] = np.mean(rep_ranks) - np.mean(dem_ranks)
    alignment[domain] = group_align

PLOT_GROUPS = ['baseline', 'dem_impl_no_img', 'rep_impl_no_img',
               'dem_img_only', 'rep_img_only', 'dem_explicit', 'rep_explicit']
PLOT_COLORS = [GRAY, DEM_LIGHT, REP_LIGHT, VIS_LIGHT, '#E1BEE7', DEM_DARK, REP_DARK]
PLOT_LABELS = ['Baseline', 'Dem Impl', 'Rep Impl',
               'Dem Img-Only', 'Rep Img-Only', 'Dem Expl', 'Rep Expl']

fig, ax = plt.subplots(1, 1, figsize=(14, 4.5))

nd = len(DOMAINS)
ng = len(PLOT_GROUPS)
x = np.arange(nd)
bar_w = 0.11
offsets = np.linspace(-(ng-1)*bar_w/2, (ng-1)*bar_w/2, ng)

for gi, g in enumerate(PLOT_GROUPS):
    vals = [alignment[d].get(g, np.nan) for d in DOMAINS]
    bars = ax.bar(x + offsets[gi], vals, bar_w,
           color=PLOT_COLORS[gi], edgecolor='white', linewidth=0.3,
           label=PLOT_LABELS[gi])
    for xi, v in enumerate(vals):
        if not np.isnan(v):
            yoff = 0.08 if v >= 0 else -0.12
            ax.text(xi + offsets[gi], v + yoff,
                     f'{v:+.1f}', ha='center', va='center', fontsize=5.5,
                     fontweight='bold', color=PLOT_COLORS[gi], alpha=0.9)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([d.title() for d in DOMAINS])
ax.set_ylabel('Rep ←  Pro-Dem Alignment (rep rank − dem rank)  → Dem', fontsize=9)
ax.set_title('Pro-Dem Preference Alignment: Zero Info to Explicit Identity', fontweight='bold')
ax.legend(ncol=4, fontsize=6.5, frameon=False, loc='upper right')

fig.tight_layout()
save_both('fig1_alignment')
plt.show()

---
## Figure 2 — Item-Level Preference: Dem Implicit vs Rep Implicit

Mean rank per item for dem_impl_no_img (blue) vs rep_impl_no_img (red). Lower rank = more preferred. D items cluster left for Dem, R items cluster left for Rep.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(20, 5), sharey=False)

cmap_demrep = LinearSegmentedColormap.from_list('dr', [DEM_DARK, '#E8E8E8', REP_DARK], N=256)

for di, domain in enumerate(DOMAINS):
    ax = axes[di]
    labels = DOMAIN_ITEMS[domain]
    m = ranking_data[domain]['matrix']
    n = len(labels)

    di_vals = m[GROUP_ORDER.index('dem_impl_no_img'), :n]
    ri_vals = m[GROUP_ORDER.index('rep_impl_no_img'), :n]
    yp = list(range(n))[::-1]

    for yi in range(n):
        d, r = di_vals[yi], ri_vals[yi]
        if not np.isnan(d) and not np.isnan(r):
            gap = abs(r - d)
            lw = max(0.5, gap * 0.5)
            ax.plot([d, r], [yp[yi], yp[yi]], color='#CCCCCC', lw=lw, zorder=1)
            ax.scatter(d, yp[yi], s=70, color=DEM_LIGHT, edgecolors='white',
                       lw=0.8, zorder=3)
            ax.scatter(r, yp[yi], s=70, color=REP_LIGHT, edgecolors='white',
                       lw=0.8, zorder=3)
            ax.text(d - 0.22, yp[yi], f'{d:.1f}', ha='right', va='center',
                    fontsize=6, color=DEM_DARK, fontweight='bold')
            ax.text(r + 0.22, yp[yi], f'{r:.1f}', ha='left', va='center',
                    fontsize=6, color=REP_DARK, fontweight='bold')

        lean = 'D' if yi < 3 else 'R'
        lc = DEM_DARK if lean == 'D' else REP_DARK
        ax.text(0.3, yp[yi], labels[yi], ha='left', va='center',
                fontsize=5.5, color=lc, fontweight='bold')

    ax.set_yticks([])
    ax.set_xlabel('Mean rank (1 = best)', fontsize=7)
    ax.set_xlim(0.1, 7.0)
    ax.invert_yaxis()
    ax.set_title(f'{chr(97+di)} {domain.title()}', loc='left', fontweight='bold',
                 fontsize=8)

fig.suptitle('Dem Implicit vs Rep Implicit: Consumer Interests Drive Divergent Preferences',
             fontweight='bold', fontsize=11, y=1.02)
fig.tight_layout(pad=1.5)
save_both('fig2_item_showdown')
plt.show()

---
## Summary

In [ ]:
print('='*80)
print('  PRO-DEM ALIGNMENT  (mean rep rank − mean dem rank; positive = pro-Dem)')
print('='*80)
print()
header = f'{"Domain":12s}'
for g in PLOT_GROUPS:
    header += f' {PLOT_LABELS[PLOT_GROUPS.index(g)]:>12s}'
print(header)
print('-' * len(header))

for domain in DOMAINS:
    row = f'{domain.title():12s}'
    for g in PLOT_GROUPS:
        v = alignment[domain].get(g, np.nan)
        row += f' {v:>12.2f}' if not np.isnan(v) else f' {"N/A":>12s}'
    print(row)

print()
for domain in DOMAINS:
    a = alignment[domain]
    impl_gap = a.get('dem_impl_no_img', 0) - a.get('rep_impl_no_img', 0)
    expl_gap = a.get('dem_explicit', 0) - a.get('rep_explicit', 0)
    face_gap = a.get('dem_img_only', 0) - a.get('rep_img_only', 0)
    print(f'{domain.title():10s} | Impl gap: {impl_gap:+.2f} | Expl gap: {expl_gap:+.2f} | Face gap: {face_gap:+.2f}')


In [ ]:
print(f'\nFigures saved to: {FIGS}')
for f in sorted(FIGS.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size // 1024} KB)')